# Data Exploration

In [158]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [159]:
RawFile = "../data/raw/automobile_dataset"
ProcessedFile = "../data/processed/automobile_dataset"

In [160]:
class fileManager():
    
    formats = {
        'csv': {'write': lambda df, f: df.to_csv(f, index=False), 'read': pd.read_csv, 'ext': '.csv'},
        'parquet': {'write': lambda df, f: df.to_parquet(f), 'read': pd.read_parquet, 'ext': '.parquet'},
        'feather': {'write': lambda df, f: df.to_feather(f), 'read': pd.read_feather, 'ext': '.feather'},
        'pickle': {'write': lambda df, f: df.to_pickle(f), 'read': pd.read_pickle, 'ext': '.pkl'}
    }

    def __init__(self,*, Format=None):
        self.format = None
        if Format is not None:
            self.setFormat(Format)
        
    @staticmethod
    def avalibleFormats() -> list:
        return list(formats.keys())
        
    @staticmethod
    def isFormat(Format: str) -> bool:
        return (Format in formats.keys())

    def setFormat(self, Format: str) -> None:
        if not self.isFormat(Format):
            raise f"{Format} format not supported. Avalible formats: {self.avalibleFormats()}"
        self.format = Format

    def getName(self, inputFilename: str) -> str:
        filename = f'{inputFilename}{formats[self.format]["ext"]}'
        return filename
        
    def read(self, inputFilename: str):
        filename = self.getName(inputFilename)
        return self.formats[self.format]["read"](filename)
        
    def write(self, df, inputFilename: str):
        filename = self.getName(inputFilename)
        return self.formats[self.format]["write"](df, filename)


In [161]:
fM = fileManager()

In [162]:
fM.setFormat("csv")
df = fM.read(RawFile)

display(df)

,Make,Model,Year,Fuel_Type,Transmission,Engine_Size,Mileage,Horsepower,Torque,Owners,Accident_History,Service_History,Color,Body_Type,Drivetrain,Fuel_Efficiency,Location,Selling_Price
0,Mercedes-Benz,GLE,2024,Petrol,Automatic,2.3,100,186.0,196.0,1,0.0,NaN,Silver,SUV,AWD,30.0,IL,64140
1,Hyundai,Tucson,2008,Petrol,Automatic,2.3,387035,189.0,190.0,4,0.0,NaN,Blue,SUV,FWD,35.0,FL,500
2,Volkswagen,Golf,2021,Hybrid,NaN,1.9,46054,158.0,153.0,1,0.0,Partial Service,Gray,Hatchback,AWD,43.0,NY,16429
3,Chevrolet,Tahoe,2005,Petrol,NaN,2.2,141302,169.0,165.0,5,1.0,No Service,Blue,SUV,AWD,NaN,FL,2199
4,Toyota,Camry,2022,Petrol,Automatic,1.9,32813,149.0,141.0,1,0.0,Full Service,Brown,Sedan,FWD,38.0,CA,21792
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5495,Volkswagen,Tiguan,2022,Petrol,Automatic,NaN,29506,161.0,NaN,1,0.0,Partial Service,Black,SUV,AWD,33.0,GA,20333
5496,Nissan,Pathfinder,2006,Petrol,Automatic,3.1,287332,243.0,227.0,5,1.0,No Service,Gray,SUV,AWD,NaN,NY,500
5497,BMW,X3,2013,Petrol,Automatic,2.2,95961,171.0,152.0,2,1.0,No Service,Blue,SUV,AWD,32.0,MI,9882
5498,Volkswagen,Golf,2020,Hybrid,Automatic,1.5,56643,135.0,125.0,2,1.0,No Service,White,Hatchback,FWD,57.0,NY,10474


In [163]:
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Data types:\n{df.dtypes}")
print()

Shape: (5500, 18)
Columns: ['Make', 'Model', 'Year', 'Fuel_Type', 'Transmission', 'Engine_Size', 'Mileage', 'Horsepower', 'Torque', 'Owners', 'Accident_History', 'Service_History', 'Color', 'Body_Type', 'Drivetrain', 'Fuel_Efficiency', 'Location', 'Selling_Price']
Data types:
Make                    str
Model                   str
Year                  int64
Fuel_Type               str
Transmission            str
Engine_Size         float64
Mileage               int64
Horsepower          float64
Torque              float64
Owners                int64
Accident_History    float64
Service_History         str
Color                   str
Body_Type               str
Drivetrain              str
Fuel_Efficiency     float64
Location                str
Selling_Price         int64
dtype: object



## Memory optimization

In [164]:
# Initial memory usage
initial_size = df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"Initial memory usage: {initial_size:.2f} MB")

# Optimize data types
for col in list(df.columns):
    if df.dtypes[col] == "int64":
        df[col] = pd.to_numeric(df[col], downcast='integer')
        size = df.memory_usage(deep=True).sum() / (1024 ** 2)
        #print(f"Memory usage after downcasting to integer: {size:.2f} MB")
    if df.dtypes[col] == "float64":
        df[col] = pd.to_numeric(df[col], downcast='float')
        size = df.memory_usage(deep=True).sum() / (1024 ** 2)
        #print(f"Memory usage after downcasting to float: {size:.2f} MB")
    if df.dtypes[col] == "str":
        df[col] = df[col].astype('category')
        size = df.memory_usage(deep=True).sum() / (1024 ** 2)
        #print(f"Memory usage after changing to categorical column: {size:.2f} MB")

# Optimized memory usage
final_size = df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"Final memory usage: {final_size:.2f} MB")
print(f"Reduction of {(1 - final_size / initial_size) * 100:.2f}%")

Initial memory usage: 1.02 MB
Final memory usage: 0.21 MB
Reduction of 79.24%


In [165]:
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Data types:\n{df.dtypes}")
print()

Shape: (5500, 18)
Columns: ['Make', 'Model', 'Year', 'Fuel_Type', 'Transmission', 'Engine_Size', 'Mileage', 'Horsepower', 'Torque', 'Owners', 'Accident_History', 'Service_History', 'Color', 'Body_Type', 'Drivetrain', 'Fuel_Efficiency', 'Location', 'Selling_Price']
Data types:
Make                category
Model               category
Year                   int16
Fuel_Type           category
Transmission        category
Engine_Size          float32
Mileage                int32
Horsepower           float32
Torque               float32
Owners                  int8
Accident_History     float32
Service_History     category
Color               category
Body_Type           category
Drivetrain          category
Fuel_Efficiency      float32
Location            category
Selling_Price          int32
dtype: object



## Missing values 

In [166]:
print("Missing values:")
print(df.isnull().sum() > 0)
print()

print("Fill missing numerical values:")
dictionary = {}

# Numerical Fixes:
dfk = df.isnull().sum() > 0
for col in list(df.columns):
    if dfk[col] and df.dtypes[col] != "category":
        dictionary[col] = dfk[col].mean()
df.fillna(dictionary)

print("Drop rows with any missing categorical values:")
print("Categorical Fixes:")
df = df.dropna()
display(df)

# Missing values:
print(df.isnull().sum() > 0)
print()

Missing values:
Make                False
Model               False
Year                False
Fuel_Type           False
Transmission         True
Engine_Size          True
Mileage             False
Horsepower           True
Torque               True
Owners              False
Accident_History     True
Service_History      True
Color                True
Body_Type           False
Drivetrain          False
Fuel_Efficiency      True
Location             True
Selling_Price       False
dtype: bool

Fill missing numerical values:
Drop rows with any missing categorical values:
Categorical Fixes:


,Make,Model,Year,Fuel_Type,Transmission,Engine_Size,Mileage,Horsepower,Torque,Owners,Accident_History,Service_History,Color,Body_Type,Drivetrain,Fuel_Efficiency,Location,Selling_Price
4,Toyota,Camry,2022,Petrol,Automatic,1.9,32813,149.0,141.0,1,0.0,Full Service,Brown,Sedan,FWD,38.0,CA,21792
7,Mercedes-Benz,C-Class,2022,Petrol,Automatic,2.3,32133,180.0,162.0,1,0.0,Partial Service,Green,Sedan,FWD,35.0,NY,31416
15,Mercedes-Benz,C-Class,2010,Petrol,Manual,2.0,17138,169.0,160.0,3,0.0,Partial Service,Blue,Sedan,FWD,27.0,NC,11728
17,Chevrolet,Tahoe,2011,Petrol,Automatic,3.3,264441,258.0,252.0,4,0.0,Partial Service,Gray,SUV,AWD,22.0,TX,3395
21,Audi,Q7,2016,Petrol,Automatic,3.1,111537,243.0,219.0,2,0.0,Partial Service,Blue,SUV,AWD,21.0,FL,23314
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5478,Audi,A6,2010,Petrol,Manual,2.1,155818,169.0,176.0,3,0.0,Full Service,Red,Sedan,FWD,41.0,FL,9714
5479,Honda,Accord,2023,Petrol,Automatic,2.4,100,199.0,192.0,1,0.0,Partial Service,Green,Sedan,FWD,33.0,GA,23937
5491,Nissan,Pathfinder,2020,Petrol,Automatic,2.7,79498,212.0,204.0,1,0.0,Partial Service,Gray,SUV,AWD,24.0,OH,18345
5497,BMW,X3,2013,Petrol,Automatic,2.2,95961,171.0,152.0,2,1.0,No Service,Blue,SUV,AWD,32.0,MI,9882


Make                False
Model               False
Year                False
Fuel_Type           False
Transmission        False
Engine_Size         False
Mileage             False
Horsepower          False
Torque              False
Owners              False
Accident_History    False
Service_History     False
Color               False
Body_Type           False
Drivetrain          False
Fuel_Efficiency     False
Location            False
Selling_Price       False
dtype: bool



## Exploration 

## Save processed data

In [167]:
fM.setFormat("parquet")
fM.write(df, ProcessedFile)

## Check it was correctly saved

In [168]:
fdf = fM.read(ProcessedFile)

display(fdf)
print(f"Shape: {fdf.shape}")
print(f"Columns: {list(fdf.columns)}")
print(f"Data types:\n{fdf.dtypes}")
print()

,Make,Model,Year,Fuel_Type,Transmission,Engine_Size,Mileage,Horsepower,Torque,Owners,Accident_History,Service_History,Color,Body_Type,Drivetrain,Fuel_Efficiency,Location,Selling_Price
4,Toyota,Camry,2022,Petrol,Automatic,1.9,32813,149.0,141.0,1,0.0,Full Service,Brown,Sedan,FWD,38.0,CA,21792
7,Mercedes-Benz,C-Class,2022,Petrol,Automatic,2.3,32133,180.0,162.0,1,0.0,Partial Service,Green,Sedan,FWD,35.0,NY,31416
15,Mercedes-Benz,C-Class,2010,Petrol,Manual,2.0,17138,169.0,160.0,3,0.0,Partial Service,Blue,Sedan,FWD,27.0,NC,11728
17,Chevrolet,Tahoe,2011,Petrol,Automatic,3.3,264441,258.0,252.0,4,0.0,Partial Service,Gray,SUV,AWD,22.0,TX,3395
21,Audi,Q7,2016,Petrol,Automatic,3.1,111537,243.0,219.0,2,0.0,Partial Service,Blue,SUV,AWD,21.0,FL,23314
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5478,Audi,A6,2010,Petrol,Manual,2.1,155818,169.0,176.0,3,0.0,Full Service,Red,Sedan,FWD,41.0,FL,9714
5479,Honda,Accord,2023,Petrol,Automatic,2.4,100,199.0,192.0,1,0.0,Partial Service,Green,Sedan,FWD,33.0,GA,23937
5491,Nissan,Pathfinder,2020,Petrol,Automatic,2.7,79498,212.0,204.0,1,0.0,Partial Service,Gray,SUV,AWD,24.0,OH,18345
5497,BMW,X3,2013,Petrol,Automatic,2.2,95961,171.0,152.0,2,1.0,No Service,Blue,SUV,AWD,32.0,MI,9882


Shape: (1329, 18)
Columns: ['Make', 'Model', 'Year', 'Fuel_Type', 'Transmission', 'Engine_Size', 'Mileage', 'Horsepower', 'Torque', 'Owners', 'Accident_History', 'Service_History', 'Color', 'Body_Type', 'Drivetrain', 'Fuel_Efficiency', 'Location', 'Selling_Price']
Data types:
Make                category
Model               category
Year                   int16
Fuel_Type           category
Transmission        category
Engine_Size          float32
Mileage                int32
Horsepower           float32
Torque               float32
Owners                  int8
Accident_History     float32
Service_History     category
Color               category
Body_Type           category
Drivetrain          category
Fuel_Efficiency      float32
Location            category
Selling_Price          int32
dtype: object

